# NeuroGolf 6225.51 Public Score Open Solution

This notebook rebuilds my current public submission for **The 2026 NeuroGolf Championship** and documents the workflow that produced it.

**Confirmed public score: 6225.51.**

The artifact is deliberately simple: one `submission.zip` containing `401` ONNX files. The notebook verifies the content manifest, writes `submission.zip`, and gives enough context to understand why this branch was promoted.

The practical lesson: high public score here came from task-level evidence, not blind blending. The current branch starts from Beicicc's strong `6202.73` public artifact, then adds four independently confirmed replacements that still transfer on that stronger base.


## What Changed Since the 5653.86 Version

The previous public notebook exposed a `5653.86` rule-mining artifact. Since then, the public landscape moved fast and a much stronger base appeared.

Current promoted chain:

| Step | Public score | Why it mattered |
|---:|---:|---|
| 5653.86 | 5653.86 | Our earlier rule-mining and public-blend branch |
| 6061.74 | 6061.74 | Beicicc public artifact showed a new hidden-transfer regime |
| 6066.71 | 6066.71 | `task305` remained a positive replacement on the 6061 base |
| 6202.73 | 6202.73 | Beicicc's newer public artifact became the correct base |
| 6225.51 | **6225.51** | Added confirmed `task100`, `task175`, `task291`, and `task305` |

The important detail is that local cost deltas did not explain the `6061 -> 6202` jump. I treated the `6202.73` artifact as the live evidence source, then only layered replacements whose task hashes proved they were not already absorbed and whose public probes transferred cleanly.


## The Current 6225.51 Blend

The promoted artifact is:

`6202.73 public base + task100 + task175 + task291 + task305`

Confirmed public deltas from the surrounding probes:

| Replacement | Public evidence |
|---|---:|
| `task100` | `6066.71 -> 6068.29` on the 6066 branch |
| `task175` | `6066.71 -> 6081.80`; also `6202.73 -> 6217.82` alone |
| `task291` | `6066.71 -> 6067.85` on the 6066 branch |
| `task305` | `6061.74 -> 6066.71` on the 6061 branch |
| Combined on 6202.73 | `6202.73 -> 6225.51` |

This is the kind of additivity I trust: narrow probes first, then a combined artifact only after the base has been checked task-by-task.


## Why This Scored Better Than a Broad Merge

A broad local-positive graft on the `6061.74` artifact changed many tasks and looked good locally, but scored only `5973.24`. That failure was useful: it showed that Beicicc's public artifacts were optimized for hidden-public behavior that my local cost estimates were not capturing.

So the workflow became stricter:

1. Use a strong public artifact as the base when it has a live score.
2. Compare task hashes before assuming a replacement is new.
3. Submit one task or a very small bundle when public behavior is uncertain.
4. Promote only tasks that survive public scoring on a near-current base.
5. Preserve the exact zip and manifest so the notebook output is auditable.

That is why this notebook publishes the artifact directly instead of trying to regenerate every ONNX graph from scratch.


## Attribution

This version builds on public evidence. The most important direct source for the base artifact is:

- `beicicc/neurogolf-6202-73-public-score-open-solution`

The broader search and comparison context also used public work from:

- `jonathanchan/ngc26-constraint-smart-logic-mix-blending`
- `svanikkolli/arc-nano-engine`
- `jazivxt/infinitesimals`
- `magmacot/5600-neurogolf-logic-driven-ensemble`

The final `6225.51` zip is not a raw repost of one source. It is the `6202.73` public base plus four replacements that were independently checked and promoted after live scoring.


## Rebuild and Verify `submission.zip`

The attached dataset contains `submission.zip`. Kaggle sometimes exposes compressed dataset files directly and sometimes exposes extracted contents depending on how the dataset is mounted, so the code handles both cases:

- if `submission.zip` exists, copy it exactly and verify its ONNX manifest;
- otherwise, collect `task*.onnx`, rebuild `submission.zip`, and verify the same manifest.

The zip SHA identifies this exact uploaded package. The content manifest is the stable identity check over all `401` ONNX files.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile

EXPECTED_FILE_COUNT = 401
EXPECTED_PUBLIC_SCORE = "6225.51"
EXPECTED_ZIP_SHA256 = "30fcf8a7a2277c217d63a94d1c782a1c5d55313bac4b94109d0dc68130788430"
EXPECTED_MANIFEST_SHA256 = "e8f7cb9fa54c18577652a7c0754c52f0ad2990155f77feefa1037cb5f1eb1fd2"
OUTPUT_ZIP = Path("submission.zip")

input_root = Path("/kaggle/input")
preferred = input_root / "neurogolf-6225-51-open-submission-artifact"
search_roots = [preferred] + sorted(input_root.glob("*6225*artifact*")) + sorted(input_root.glob("*neurogolf*artifact*"))


def manifest_from_zip(zip_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(name for name in zf.namelist() if name.endswith(".onnx"))
        lines = []
        for name in names:
            data = zf.read(name)
            lines.append(f"{Path(name).name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    return names, hashlib.sha256("\n".join(lines).encode("utf-8")).hexdigest()

source_zip = None
source_dir = None
for root in search_roots:
    if not root.exists():
        continue
    zips = sorted(root.glob("*.zip"))
    if zips:
        source_zip = zips[0]
        break
    onnx_files = sorted(root.glob("task*.onnx"))
    if len(onnx_files) == EXPECTED_FILE_COUNT:
        source_dir = root
        break

assert source_zip is not None or source_dir is not None, "Could not find the 6225.51 artifact under /kaggle/input"

if source_zip is not None:
    shutil.copyfile(source_zip, OUTPUT_ZIP)
    names, manifest_sha = manifest_from_zip(OUTPUT_ZIP)
else:
    onnx_files = sorted(source_dir.glob("task*.onnx"))
    with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in onnx_files:
            zf.write(path, arcname=path.name)
    names, manifest_sha = manifest_from_zip(OUTPUT_ZIP)

zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
print("onnx_file_count:", len(names))
print("first_file:", names[0] if names else None)
print("last_file:", names[-1] if names else None)
print("manifest_sha256:", manifest_sha)
print("zip_sha256:", zip_sha)
print("submission_zip_bytes:", OUTPUT_ZIP.stat().st_size)
print("expected_public_score:", EXPECTED_PUBLIC_SCORE)

assert len(names) == EXPECTED_FILE_COUNT, len(names)
assert manifest_sha == EXPECTED_MANIFEST_SHA256, manifest_sha
if source_zip is not None:
    assert zip_sha == EXPECTED_ZIP_SHA256, zip_sha
print("Artifact check passed. submission.zip is ready.")


## Practical Takeaways

- A high public artifact is a base, not an answer key. It still needs task-level auditing.
- Local score can be misleading when the public artifact was selected for hidden transfer.
- Small probes are not slow in practice; they prevent large ambiguous failures.
- Hash comparisons matter. If the new base already changed a task, do not blindly reapply an old replacement.
- For this competition, a clean artifact notebook can be more useful than a long opaque solver when the goal is reproducibility and comparison.

I will keep updating this same notebook when a stronger public-scored artifact is confirmed, so readers can follow the current best branch without chasing multiple copies.
